In [0]:
# Install required libraries for this notebook
%pip install xgboost shap 

In [0]:
# Notebook 6: Restaurant Health Scoring
# Food Delivery Analysis
# ----------------------------------------

import mlflow
import mlflow.xgboost
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# Load restaurant features from Parquet
restaurant_features = spark.read.parquet(
    "/Volumes/workspace/default/food_delivery_data/restaurant_features.parquet"
)

print(f"Loaded {restaurant_features.count():,} restaurants with {len(restaurant_features.columns)} features.")
restaurant_features.display()

print(f"Total restaurants: {restaurant_features.count():,}")
print(f"Columns: {restaurant_features.columns}")

In [0]:
# Convert to Pandas for modelling
restaurant_pd = restaurant_features.toPandas()

# Create binary health label
# Unhealthy = restaurant_health_score below 0.6
# Healthy = restaurant_health_score 0.6 and above
restaurant_pd["is_unhealthy"] = (restaurant_pd["restaurant_health_score"] < 0.6).astype(int)

# Drop columns not needed for modelling
restaurant_pd = restaurant_pd.drop(columns=[
    "restaurant_id", "restaurant_name",
    "top_ordered_item", "restaurant_health_score"
])

# Encode text columns
le = LabelEncoder()
for col_name in ["cuisine", "city", "area"]:
    restaurant_pd[col_name] = le.fit_transform(restaurant_pd[col_name])

# Separate features and target
X = restaurant_pd.drop(columns=["is_unhealthy"])
y = restaurant_pd["is_unhealthy"]

print(f"Features shape: {X.shape}")
print(f"Unhealthy restaurant rate: {y.mean():.1%}")
print(f"\nFeature columns:\n{list(X.columns)}")

In [0]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} restaurants")
print(f"Test set: {X_test.shape[0]:,} restaurants")
print(f"\nUnhealthy rate in training set: {y_train.mean():.1%}")
print(f"Unhealthy rate in test set: {y_test.mean():.1%}")

In [0]:
# Train XGBoost restaurant health scoring model
with mlflow.start_run(run_name="XGBoost Restaurant Health Scoring"):

    xgb_model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss",
        use_label_encoder=False
    )

    xgb_model.fit(X_train, y_train)

    y_pred = xgb_model.predict(X_test)
    y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.xgboost.log_model(xgb_model, "xgboost_restaurant_health_model")

    print(f"Accuracy:  {accuracy:.1%}")
    print(f"ROC AUC:   {roc_auc:.3f}")
    print(f"\nDetailed Report:")
    print(classification_report(y_test, y_pred,
          target_names=["Healthy", "Unhealthy"]))

In [0]:
# Confusion matrix for restaurant health scoring model
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds",
            xticklabels=["Healthy", "Unhealthy"],
            yticklabels=["Healthy", "Unhealthy"])
plt.title("XGBoost Restaurant Health Scoring - Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("/tmp/xgb_restaurant_health_confusion_matrix.png", dpi=150)
plt.show()
print("Confusion matrix saved.")

In [0]:
# SHAP feature importance for restaurant health scoring model
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title("XGBoost Restaurant Health Scoring - Feature Importance (SHAP)")
plt.tight_layout()
plt.savefig("/tmp/xgb_restaurant_health_shap.png", dpi=150)
plt.show()
print("SHAP feature importance saved.")

In [0]:
# Restaurant health scoring summary
print("-" * 55)
print("RESTAURANT HEALTH SCORING SUMMARY")
print("-" * 55)
print(f"{'Metric':<30} {'Value':>10}")
print("-" * 55)
print(f"{'Accuracy':<30} {'97.2%':>10}")
print(f"{'ROC AUC':<30} {'0.993':>10}")
print(f"{'Healthy Recall':<30} {'0.98':>10}")
print(f"{'Unhealthy Recall':<30} {'0.96':>10}")
print(f"{'Unhealthy Restaurants Detected':<30} {'88/92':>10}")
print(f"{'False Alarms':<30} {'7/308':>10}")
print("-" * 55)
print("""
The model correctly identifies 96% of underperforming 
restaurants while raising false alarms on only 7 healthy 
restaurants. The strongest predictor of restaurant health 
is the average order quality risk score, followed by 
average delivery duration and cancellation rate.

These results confirm that restaurant underperformance is 
driven primarily by operational factors, delivery speed 
and order reliability, rather than cuisine type or location.
""")

In [0]:
# Logistic Regression for comparison
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Logistic Regression requires feature scaling unlike tree based models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

with mlflow.start_run(run_name="Logistic Regression Restaurant Health"):

    lr_model = LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight="balanced"
    )

    lr_model.fit(X_train_scaled, y_train)

    y_pred_lr = lr_model.predict(X_test_scaled)
    y_pred_lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

    accuracy_lr = accuracy_score(y_test, y_pred_lr)
    roc_auc_lr = roc_auc_score(y_test, y_pred_lr_proba)

    mlflow.log_metric("accuracy", accuracy_lr)
    mlflow.log_metric("roc_auc", roc_auc_lr)

    print(f"Accuracy:  {accuracy_lr:.1%}")
    print(f"ROC AUC:   {roc_auc_lr:.3f}")
    print(f"\nDetailed Report:")
    print(classification_report(y_test, y_pred_lr,
          target_names=["Healthy", "Unhealthy"]))

In [0]:
# Support Vector Machine for comparison
from sklearn.svm import SVC

with mlflow.start_run(run_name="SVM Restaurant Health"):

    svm_model = SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        probability=True,
        random_state=42,
        class_weight="balanced"
    )

    # SVM also requires scaled features
    svm_model.fit(X_train_scaled, y_train)

    y_pred_svm = svm_model.predict(X_test_scaled)
    y_pred_svm_proba = svm_model.predict_proba(X_test_scaled)[:, 1]

    accuracy_svm = accuracy_score(y_test, y_pred_svm)
    roc_auc_svm = roc_auc_score(y_test, y_pred_svm_proba)

    mlflow.log_metric("accuracy", accuracy_svm)
    mlflow.log_metric("roc_auc", roc_auc_svm)

    print(f"Accuracy:  {accuracy_svm:.1%}")
    print(f"ROC AUC:   {roc_auc_svm:.3f}")
    print(f"\nDetailed Report:")
    print(classification_report(y_test, y_pred_svm,
          target_names=["Healthy", "Unhealthy"]))

In [0]:
# Ensemble model combining XGBoost and SVM using soft voting
# XGBoost uses original features, SVM uses scaled features

with mlflow.start_run(run_name="XGBoost SVM Ensemble Restaurant Health"):

    # Get probability predictions from both models
    xgb_proba  = xgb_model.predict_proba(X_test)[:, 1]
    svm_proba  = svm_model.predict_proba(X_test_scaled)[:, 1]

    # Average the probabilities from both models
    ensemble_proba = (xgb_proba + svm_proba) / 2

    # Convert averaged probability to final class prediction
    ensemble_pred = (ensemble_proba >= 0.5).astype(int)

    accuracy_ens = accuracy_score(y_test, ensemble_pred)
    roc_auc_ens  = roc_auc_score(y_test, ensemble_proba)

    mlflow.log_metric("accuracy", accuracy_ens)
    mlflow.log_metric("roc_auc", roc_auc_ens)

    print(f"Accuracy:  {accuracy_ens:.1%}")
    print(f"ROC AUC:   {roc_auc_ens:.3f}")
    print(f"\nDetailed Report:")
    print(classification_report(y_test, ensemble_pred,
          target_names=["Healthy", "Unhealthy"]))

In [0]:
# Full model comparison summary including ensemble
print("-" * 70)
print("RESTAURANT HEALTH SCORING - FULL MODEL COMPARISON")
print("-" * 70)
print(f"{'Model':<30} {'Accuracy':>10} {'ROC AUC':>10} {'Recall':>10}")
print("-" * 70)
print(f"{'XGBoost':<30} {'97.2%':>10} {'0.993':>10} {'0.96':>10}")
print(f"{'Logistic Regression':<30} {'95.0%':>10} {'0.995':>10} {'0.98':>10}")
print(f"{'SVM':<30} {'92.8%':>10} {'0.992':>10} {'0.99':>10}")
print(f"{'XGBoost + SVM Ensemble':<30} {'97.0%':>10} {'0.994':>10} {'0.96':>10}")
print("-" * 70)
print("""
XGBoost achieves the best overall accuracy and the strongest 
balance between precision and recall, making it the preferred 
model for production deployment.

The XGBoost and SVM ensemble matches XGBoost almost exactly, 
demonstrating that ensembling adds value only when the base 
models make meaningfully different errors. In this case both 
models are already strong, so the ensemble provides marginal 
improvement.

Logistic Regression performs competitively despite being a 
simple linear model, confirming that restaurant health is 
driven by strong linear operational signals.

Selected model for deployment: XGBoost
""")